# Stickman Motion Studio — ComfyUI server smoke test

This stage installs ComfyUI in the Kaggle session, starts its local API without downloading a checkpoint, and verifies that the server responds while PyTorch still sees CUDA. It does not generate an image yet.


In [ ]:
import os, sys, subprocess, pathlib, json, time, urllib.request, urllib.error
print('Python:', sys.version)
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA devices:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), 'CUDA is not available in this Kaggle session.'
repo = pathlib.Path('/kaggle/working/ComfyUI')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(repo)], check=True)
print('ComfyUI checkout:', repo)


## Install ComfyUI Python requirements

The installer uses the Kaggle environment's existing PyTorch/CUDA packages where compatible. It does not download model checkpoints.


In [ ]:
requirements = (repo / 'requirements.txt').read_text(encoding='utf-8').splitlines()
import re
skip = {'torch', 'torchvision', 'torchaudio'}
filtered = []
for line in requirements:
    stripped = line.strip()
    if not stripped or stripped.startswith('#'):
        continue
    match = re.match(r'^([A-Za-z0-9_.-]+)', stripped)
    package_name = match.group(1).lower().replace('_', '-') if match else ''
    if package_name not in skip:
        filtered.append(line)
tmp_req = pathlib.Path('/kaggle/working/comfyui-kaggle-requirements.txt')
tmp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
print('Installing ComfyUI requirements while preserving Kaggle torch packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(tmp_req)], check=True)
import torch
print('After install — PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA was lost after dependency installation.'


## Start ComfyUI and test its API

No checkpoint is required for this server/API test. The next stage will add a small compatible model and perform a real image-generation test.


In [ ]:
import subprocess, time, urllib.request, json, pathlib, os, signal
log_path = pathlib.Path('/kaggle/working/comfyui-server.log')
log_file = log_path.open('w')
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd=repo, stdout=log_file, stderr=subprocess.STDOUT)
base = 'http://127.0.0.1:8188'
stats = None
try:
    deadline = time.time() + 240
    last_error = None
    while time.time() < deadline:
        if server.poll() is not None:
            break
        try:
            with urllib.request.urlopen(base + '/system_stats', timeout=5) as response:
                stats = json.loads(response.read().decode('utf-8'))
            break
        except Exception as exc:
            last_error = repr(exc)
            time.sleep(3)
    if stats is None:
        print('ComfyUI server log (last 12000 chars):')
        log_file.flush()
        print(log_path.read_text(encoding='utf-8', errors='replace')[-12000:])
        raise RuntimeError('ComfyUI API did not become ready. Last request error: ' + str(last_error))
    with urllib.request.urlopen(base + '/object_info', timeout=30) as response:
        nodes = json.loads(response.read().decode('utf-8'))
    report = {
        'run_token': globals().get('RUN_TOKEN', 'manual'),
        'comfyui_started': True,
        'api_system_stats_ok': True,
        'api_object_info_ok': True,
        'node_type_count': len(nodes),
        'torch_version': torch.__version__,
        'cuda_available': torch.cuda.is_available(),
        'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        'system_stats': stats
    }
    pathlib.Path('/kaggle/working/comfyui-smoke-test.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps({k:v for k,v in report.items() if k != 'system_stats'}, indent=2))
    print('Saved /kaggle/working/comfyui-smoke-test.json')
finally:
    if server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=20)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()
    log_file.close()
